# 06 Web scraping: extract the 1500 m results into fields

## Purpose of this notebook

This notebook applies the same web scraping mechanics to an athletics results page.

Source page: <https://www.alltime-athletics.com/w_1500ok.htm>

The task is deliberately narrow:

1. fetch the page
2. inspect the HTML structure
3. find the correct tag that holds the results
4. extract the text lines from that tag
5. use a fixed-width converter to turn the lines into fields
6. save the fielded data as a CSV file

We do not analyse the data in this notebook. We do not calculate times, convert dates, group results, or make charts. We only extract the fields that are already visible on the page.

No regular expressions are used.


## Before coding: inspect the results page

Before you run the Python code, open the source page in a browser:

<https://www.alltime-athletics.com/w_1500ok.htm>

Spend a few minutes checking the page as a person first:

- What event does the page describe?
- Where do the results appear on the page?
- Do the results look like a normal HTML table, or do they look like aligned text?
- What fields can you see in one result line?

Then check the HTML behind the page. You can use **View page source** or the browser's **Inspect** tool.

Look for the tag that contains the results. On this page, the results are stored inside a `<pre>` tag. A `<pre>` tag keeps spaces and line breaks, which is why the result rows line up like a text table.

This inspection explains why the code later uses BeautifulSoup to find `<pre>` tags and `pd.read_fwf()` to convert the aligned text into fields.


## Step 1: import the libraries

`requests` downloads the page.

`BeautifulSoup` reads the HTML and lets us search for the tag that contains the results.

`pandas` is used to convert the fixed-width text into fields and save the CSV file.


In [ ]:
from pathlib import Path
import io

import pandas as pd
import requests
from bs4 import BeautifulSoup


## Step 2: fetch the page

This page is an older HTML page. We make one request and check the status code before doing anything else.


In [ ]:
url = "https://www.alltime-athletics.com/w_1500ok.htm"
headers = {
    "User-Agent": "VIT1106 teaching example (one polite request)"
}

response = requests.get(url, headers=headers, timeout=30)
print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))


## Step 3: parse the HTML

BeautifulSoup lets us inspect the tags in the page.


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

page_title = soup.find("title")
if page_title is not None:
    print(page_title.get_text(" ", strip=True))


## Step 4: find the tag that holds the results

After inspecting the page structure, the results list is stored in a `<pre>` tag.

A `<pre>` tag means preformatted text. It keeps spaces and line breaks, which is why the results line up like a text table.

This page has more than one `<pre>` tag, so we first count them and preview their text.


In [ ]:
pre_tags = soup.find_all("pre")
print("Number of pre tags:", len(pre_tags))

for number, tag in enumerate(pre_tags, start=1):
    preview = tag.get_text().strip().splitlines()[:3]
    print()
    print("pre tag", number)
    for line in preview:
        print(line[:120])


## Step 5: select the results tag

The first `<pre>` tag contains the women's 1500 m results. We will extract its text exactly as the page gives it to us.


In [ ]:
results_tag = pre_tags[0]
raw_text = results_tag.get_text()

lines = raw_text.splitlines()
print("Total lines in tag:", len(lines))
print("First non-empty lines:")

shown = 0
for line in lines:
    clean_line = line.strip()
    if clean_line:
        print(clean_line[:120])
        shown = shown + 1
    if shown == 5:
        break


## Step 6: keep the non-empty result lines

We remove blank lines so the fixed-width converter receives only result records.

We are not cleaning the values yet. We are only preparing the lines so they can be read as fields.


In [ ]:
result_lines = []

for line in lines:
    if line.strip():
        result_lines.append(line)

print("Result lines:", len(result_lines))
result_lines[:5]


## Step 7: convert the fixed-width text into fields with pandas

The results are stored as **fixed-width text**. This means the values line up in regular positions, like columns in a plain text table.

For example, the rank starts near the beginning of the line, the time comes next, the athlete name has its own space, and the country code appears after the name.

`pandas.read_fwf()` means **read fixed-width file**. It is a pandas converter for text where the columns are separated by position rather than by commas.

We use it here because the page does not give us a normal HTML table or a comma-separated CSV file. The data is inside a `<pre>` tag, so we first collect the text lines, then ask pandas to split the aligned text into fields.

The construct `io.StringIO(text_block)` may look unusual. It lets pandas read a text string as if it were a small file. This is useful because `pd.read_fwf()` expects file-like input, but our web scraping code has collected the results as text in memory.

Before running the next cell, ask AI to explain this line in beginner language:

> In this notebook, what does `io.StringIO(text_block)` do, and why is it used with `pd.read_fwf()`? Please explain it for a first-year data science student.

This is still extraction, not analysis. We are naming the fields so the CSV is useful later.


In [ ]:
text_block = "\n".join(result_lines)

field_names = [
    "rank",
    "mark",
    "athlete",
    "country",
    "date_of_birth",
    "race_position",
    "venue",
    "result_date",
]

column_positions = [
    (0, 15),
    (15, 26),
    (26, 57),
    (57, 65),
    (65, 77),
    (77, 84),
    (84, 112),
    (112, None),
]

df_results = pd.read_fwf(
    io.StringIO(text_block),
    header=None,
    colspecs=column_positions,
    names=field_names,
    dtype=str,
    keep_default_na=False,
)

print("Rows:", len(df_results))
print("Columns:", list(df_results.columns))
df_results.head(10)


## Step 8: the same field split with introductory Python

`pd.read_fwf()` is the cleaner pandas tool for fixed-width text. The next cell shows the same idea using introductory Python techniques.

This version uses:

- a `for` loop
- string slicing such as `line[0:15]`
- `.strip()` to remove extra spaces
- dictionaries to label the fields
- a list to collect the rows
- `pd.DataFrame()` to make the table

This is useful for learning because it makes the column positions visible. In normal work, you would usually keep the shorter `pd.read_fwf()` version above.


In [ ]:
manual_rows = []

for line in result_lines:
    row = {
        "rank": line[0:15].strip(),
        "mark": line[15:26].strip(),
        "athlete": line[26:57].strip(),
        "country": line[57:65].strip(),
        "date_of_birth": line[65:77].strip(),
        "race_position": line[77:84].strip(),
        "venue": line[84:112].strip(),
        "result_date": line[112:].strip(),
    }
    manual_rows.append(row)

df_manual_results = pd.DataFrame(manual_rows, columns=field_names)

print("Rows:", len(df_manual_results))
print("Columns:", list(df_manual_results.columns))
print("Same result as pd.read_fwf():", df_manual_results.equals(df_results))
df_manual_results.head(10)


## Step 9: save the fielded CSV file

Now save the fielded extraction as a CSV file. We save the pandas `read_fwf()` result because it is the cleaner method for this type of text.


In [ ]:
DATA_FOLDER = Path("data")
DATA_FOLDER.mkdir(exist_ok=True)

OUTPUT_FILE = DATA_FOLDER / "w1500_alltime_fields.csv"
df_results.to_csv(OUTPUT_FILE, index=False)

print("Saved", len(df_results), "rows to", OUTPUT_FILE)


## What you learned

In this notebook, you practised a clean extraction workflow:

- fetch one page
- parse the HTML with BeautifulSoup
- find the tag that contains the data
- extract the tag text
- keep the result lines
- use `pd.read_fwf()` to split fixed-width text into fields
- use `io.StringIO()` when pandas needs to read text that is already in memory
- use basic Python slicing to see how fixed-width fields work
- save the fielded extraction as CSV

The key habit is separation. Extraction and fielding are one job. Cleaning, conversion, and analysis are later jobs.
